# Data Loader

**Requirements**:

To run this notebook entirely you need an API key from the 'Chicago Data Portal' (https://data.cityofchicago.org/). However, you can skip the API call, therefore please set **SAMPLE_MODE** to True.

**Description**:

This notebook can run in two modes: **SAMPLE_MODE**. If sample_mode is set to be true the notebook will run with a sample of 5,000 rows of the taxi data. 

This notebook loads the data sources via API or stores data provided under /raw_data to .parquet files. The .parquet files generate by this notebook are referred to as "bronze" level.

Further this notebooks loads POI data from OpenStreetMap via OSMnx and stores the data as parquet file.

## List of Used Data Sources

**Primary Data Sources**

| Data Source                  | Description                                                       | Link                                                                                                                                                                                                                                                        |
| ---------------------------- | ----------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Taxi Data                    | Chicago Taxi Trips 2024, accessed via API                         | [Taxi Trips 2024](https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data)                                                                                                                                                      |
| Census Tract Data            | Geographic boundaries for Chicago census tracts                   | [Census Tracts](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Census_Tracts/4hp8-2i8z/about_data)                                                                                                                                         |
| Community Areas              | Geographic boundaries for Chicago community areas                 | [Boundaries - Community Areas](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas/igwz-8jzy/about_data)                                                                                                             |                                                                                                                                                            |
| Weather Data - MDW           | Weather observations for Chicago Midway Airport `MDW`             | [MDW ASOS Data Query](https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=MDW&data=all&year1=2024&month1=1&day1=1&year2=2026&month2=5&day2=26&tz=America%2FChicago&format=onlycomma&latlon=yes&elev=yes&missing=null&trace=0.0001&direct=no) |                                                                                                                                       |                                                                                                     |

Link for weather data without query setting: https://mesonet.agron.iastate.edu/request/download.phtml?network=IL_ASOS

Point-of-Interest data was downloaded via OSMnx from OpenStreetMap

Further we use the package "holidays" to get the holidays relevant for chicago.

**Data Pipeline**

```text
SODA API
   ↓
CSV as Raw Backup
   ↓
Parquet
   ↓
Polars LazyFrame & DuckDB
```

**Data Preparation Pipeline**

```text
Raw CSV
   ↓
Bronze Parquet
   1:1 from CSV/API, kept as unchanged as possible
   ↓
Silver Parquet
   Cleaned, typed, and filtered data
   ↓
Gold / Features
   Merged and aggregated datasets, ML features, train/test-ready datasets
```

**Pipeline Layers**

| Layer           | Description                                                                  |
| --------------- | ---------------------------------------------------------------------------- |
| Raw CSV         | Original data export or API response stored as CSV backup                    |
| Bronze Parquet  | One-to-one conversion from CSV/API, kept as unchanged as possible            |
| Silver Parquet  | Cleaned, typed, and filtered version of the data                             |
| Gold / Features | Feature-engineered datasets ready for machine learning and train/test splits |



In [ ]:
# TODO add holidays

# Imports and Settings

In [ ]:
import pandas as pd
import requests
from pathlib import Path
import duckdb

import polars as pl
import h3
import geopandas as gpd

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import osmnx as ox
import pandas as pd

from shapely.geometry import Polygon, MultiPolygon

In [18]:
SAMPLE_MODE = True
APP_TOKEN = "***"

START_DATE = "2024-01-01T00:00:00"
END_DATE = "2026-05-01T00:00:00"

if SAMPLE_MODE:
    print("Runs in SAMPLE_MODE with APP_TOKEN = ", APP_TOKEN)
    
else:
    print("Attention: Runs NOT in SAMPLE_MODE with APP_TOKEN = ", APP_TOKEN)

Runs in SAMPLE_MODE with APP_TOKEN =  ***


## Download Taxi Data via API
Load the data from "https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data" using SODA3 API and store it as CSV.

In [ ]:
# API Parameter
DATASET_ID = "ajtu-isnz"
API_URL = f"https://data.cityofchicago.org/api/v3/views/ajtu-isnz/query.csv"

OUTPUT_PATH = Path("../data/raw_data/taxi_sample.csv") if SAMPLE_MODE else Path("../data/raw_data/taxi.csv")

# Query Parameter
LIMIT = 5_000 if SAMPLE_MODE else None


query = f"""
SELECT
    *
WHERE
    trip_start_timestamp >= '{START_DATE}'
    AND trip_start_timestamp < '{END_DATE}'
LIMIT {LIMIT}
"""

query = query if LIMIT is not None else f"""SELECT * WHERE trip_start_timestamp >= '{START_DATE}'
    AND trip_start_timestamp < '{END_DATE}'"""

# Build payload
headers = {}

if APP_TOKEN and APP_TOKEN != "***":
    headers["X-App-Token"] = APP_TOKEN

payload = {
    "query": query
}

# CSV per Post streamen
with requests.post(
    API_URL,
    headers=headers,
    json=payload,
    stream=True,
    timeout=180
) as r:
    r.raise_for_status()

    with OUTPUT_PATH.open("wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print(f"Download finished: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")

Download finished: ../data/raw_data/taxi_small.csv
File size: 4.87 MB


## Download POI via OpenStreetMap

In [14]:
PLACE = "Chicago, Illinois, USA"


POI_TAGS: dict[str, dict[str, list[str] | str | bool]] = {
    # Restaurants / Cafés / Bars etc.
    "food_drink": {
        "amenity": [
            "restaurant",
            "cafe",
            "fast_food",
            "bar",
            "pub",
            "food_court",
            "ice_cream",
        ],
    },

    # Rail / CTA / Metra-nahe OSM-Features.
    # Bewusst NICHT public_transport=platform oder bus_stop, sonst wird es sehr viel.
    "train_station": {
        "railway": [
            "station",
            "halt",
            "subway_entrance",
            "tram_stop",
        ],
    },

    # Shops: bewusst eingeschränkt, damit es nicht zu granular wird.
    "shop": {
        "shop": [
            "supermarket",
            "convenience",
            "mall",
            "department_store",
            "clothes",
            "bakery",
            "pharmacy",
            "electronics",
        ],
    },

    # Wahrzeichen / touristisch relevante Orte.
    "landmark": {
        "tourism": [
            "attraction",
            "museum",
            "gallery",
            "viewpoint",
            "zoo",
            "aquarium",
        ],
        "historic": [
            "monument",
            "memorial",
        ],
        "man_made": [
            "tower",
            "lighthouse",
        ],
        "amenity": [
            "theatre",
            "arts_centre",
            "cinema",
        ],
    },
}


def configure_osmnx(project_root: Path) -> None:
    """
    OSMnx-Settings: Cache aktivieren, damit du Overpass nicht unnötig oft belastest.
    """
    cache_dir = project_root / ".cache" / "osmnx"
    cache_dir.mkdir(parents=True, exist_ok=True)

    ox.settings.use_cache = True
    ox.settings.cache_folder = str(cache_dir)
    ox.settings.log_console = True
    ox.settings.requests_timeout = 180


def fetch_poi_category(
    place: str,
    category: str,
    tags: dict[str, list[str] | str | bool],
) -> gpd.GeoDataFrame:
    """
    Lädt eine POI-Kategorie aus OpenStreetMap.
    """
    print(f"Fetching {category} ...")

    gdf = ox.features_from_place(place, tags=tags)

    if gdf.empty:
        return gpd.GeoDataFrame(columns=["poi_category", "geometry"], geometry="geometry", crs="EPSG:4326")

    gdf = gdf.reset_index()
    gdf["poi_category"] = category

    return gdf


def add_point_geometry(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    OSM-Features können Punkte, Polygone oder Linien sein.
    Für ML/Feature Engineering mit Taxi-Daten ist oft ein Punkt pro POI praktischer.
    Daher wird aus jeder Geometrie ein repräsentativer Punkt erzeugt.
    """
    gdf = gdf.copy()

    gdf["geom_type_original"] = gdf.geometry.geom_type

    # Für Flächenberechnung und representative_point besser in ein metrisches CRS projizieren.
    metric_crs = gdf.estimate_utm_crs()
    gdf_metric = gdf.to_crs(metric_crs)

    gdf_metric["area_m2"] = gdf_metric.geometry.area

    # Für Points bleibt representative_point identisch bzw. sinnvoll.
    gdf_metric["geometry"] = gdf_metric.geometry.representative_point()

    gdf_points = gdf_metric.to_crs("EPSG:4326")

    gdf_points["lon"] = gdf_points.geometry.x
    gdf_points["lat"] = gdf_points.geometry.y

    return gdf_points


def clean_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Spalten reduzieren und Namen parquet-/polars-freundlicher machen.
    """
    keep_cols = [
        "element_type",
        "osmid",
        "poi_category",
        "name",
        "amenity",
        "shop",
        "railway",
        "tourism",
        "historic",
        "man_made",
        "brand",
        "operator",
        "opening_hours",
        "addr:housenumber",
        "addr:street",
        "addr:city",
        "website",
        "phone",
        "geom_type_original",
        "area_m2",
        "lat",
        "lon",
        "geometry",
    ]

    existing_cols = [col for col in keep_cols if col in gdf.columns]
    gdf = gdf[existing_cols].copy()

    # Doppelpunkte sind in Geo/OSM üblich, aber für spätere Verarbeitung oft nervig.
    rename_map = {
        col: col.replace(":", "_").replace("-", "_")
        for col in gdf.columns
        if col != "geometry"
    }
    gdf = gdf.rename(columns=rename_map)

    return gdf


def deduplicate_pois(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Ein OSM-Objekt kann in mehreren Kategorien landen.
    Hier werden Duplikate über element_type + osmid entfernt.
    """
    gdf = gdf.copy()

    if "element_type" not in gdf.columns or "osmid" not in gdf.columns:
        return gdf.drop_duplicates()

    gdf["osm_uid"] = gdf["element_type"].astype(str) + "/" + gdf["osmid"].astype(str)

    # Wenn ein POI mehrfach vorkommt, Kategorien zusammenführen.
    rows = []
    for _, group in gdf.groupby("osm_uid", sort=False):
        row = group.iloc[0].copy()
        row["poi_category"] = "|".join(sorted(group["poi_category"].dropna().astype(str).unique()))
        rows.append(row)

    result = gpd.GeoDataFrame(rows, geometry="geometry", crs=gdf.crs)
    return result.drop(columns=["osm_uid"])


def extract_chicago_pois(
    place: str = PLACE,
    poi_tags: dict[str, dict[str, list[str] | str | bool]] = POI_TAGS,
) -> gpd.GeoDataFrame:
    frames: list[gpd.GeoDataFrame] = []

    for category, tags in poi_tags.items():
        gdf_category = fetch_poi_category(place=place, category=category, tags=tags)
        if not gdf_category.empty:
            frames.append(gdf_category)

    if not frames:
        raise ValueError("No POIs found. Check place name or OSM tags.")

    pois = pd.concat(frames, ignore_index=True)
    pois = gpd.GeoDataFrame(pois, geometry="geometry", crs=frames[0].crs)

    pois = deduplicate_pois(pois)
    pois = add_point_geometry(pois)
    pois = clean_columns(pois)

    return pois

In [16]:
project_root = Path("../")
configure_osmnx(project_root)

processed_dir = project_root / "data" / "processed_data"
processed_dir.mkdir(parents=True, exist_ok=True)

geo_out = processed_dir / "bronze_osm_chicago_pois.geoparquet"
tabular_out = processed_dir / "bronze_osm_chicago_pois.parquet"

pois = extract_chicago_pois()

# GeoParquet: Geometrie bleibt erhalten.
pois.to_parquet(geo_out, index=False)

# Normales Parquet für Polars/ML: lat/lon bleiben, geometry wird entfernt.
pois.drop(columns="geometry").to_parquet(tabular_out, index=False)

print(f"Saved GeoParquet: {geo_out}")
print(f"Saved tabular Parquet: {tabular_out}")
print("\nCounts by category:")
print(pois["poi_category"].value_counts())

Fetching food_drink ...
Fetching train_station ...
Fetching shop ...
Fetching landmark ...
Saved GeoParquet: ../data/processed_data/bronze_osm_chicago_pois.geoparquet
Saved tabular Parquet: ../data/processed_data/bronze_osm_chicago_pois.parquet

Counts by category:
poi_category
food_drink       5791
shop             1783
landmark          973
train_station     570
Name: count, dtype: int64


## Create Parquet File for efficient Data Handling

In [12]:
overwrite = False

raw_dir = Path("../data/raw_data")
processed_dir = Path("../data/processed_data")

if not raw_dir.exists():
    raise FileNotFoundError(f"Raw data folder does not exist: {raw_dir}")

processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(raw_dir.glob("*.csv"))

if not csv_files:
    print(f"No CSV files found in {raw_dir}")

for csv_path in csv_files:
    parquet_path = processed_dir / f"bronze_{csv_path.stem}.parquet"

    if parquet_path.exists() and not overwrite:
        print(f"Skipping {csv_path.name}: {parquet_path.name} already exists")
        
        print("Quality check - Rows in parquet file:")
    
        duckdb.sql(f"""
        SELECT count(*)
        FROM read_parquet('{parquet_path}')
        """).show()
    
        continue

    print(f"Converting {csv_path.name} -> {parquet_path.name}")
    
    duckdb.sql(f"""
    COPY (
        SELECT *
        FROM read_csv_auto(
            '{csv_path}',
            header = true,
            sample_size = -1
        )
    )
    TO '{parquet_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    );
    """)
    
    print("Quality check - Rows in parquet file:")
    
    duckdb.sql(f"""
    SELECT count(*)
    FROM read_parquet('{parquet_path}')
    """).show()



print("Done creating parquet files from csv.")

Skipping Census_Tracts.csv: bronze_Census_Tracts.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          878 │
└──────────────┘

Converting Community_Areas.csv -> bronze_Community_Areas.parquet
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           77 │
└──────────────┘

Skipping Individual_Landmarks.csv: bronze_Individual_Landmarks.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          407 │
└──────────────┘

Skipping taxi.csv: bronze_taxi.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     15406960 │
└──────────────┘

Converting taxi_1000.csv -> bronze_taxi_1000.parquet
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
